In [1]:
import os
import sys
import scipp as sc
import mcstastox as mx

parent = os.path.dirname(os.getcwd())
sys.path.append(parent)


van_path = parent + "/runs/LET_vanad_huge/LET_vanad_square_0"

with mx.Read(van_path) as mcstas_van_data:
    scipp_van_data_group = mcstas_van_data.export_scipp(
        source_name="SourceMantid",
        sample_name="iso_samp",
    )

v_events_binned = scipp_van_data_group["events"]
display(v_events_binned)
# Group by pixel_id to fill in the pixels that recorded no events
det_positions = scipp_van_data_group["positions"]
sample_position = v_events_binned.coords["sample_position"]
v_events = v_events_binned.group(sc.arange(det_positions.dim, 0, det_positions.size))
display(v_events)


<scipp.DataArray>
Dimensions: Sizes[pixel_id:98300, ]
Coordinates:
* pixel_id                    int64  [dimensionless]  (pixel_id)  [0, 1, ..., 98302, 98303]
* position                  vector3              [m]  (pixel_id)  [(-2.23877, -1.99219, 27.6903), (-2.21669, -1.99219, 27.7086), ..., (2.28249, 1.99219, 22.3467), (2.26071, 1.99219, 22.3281)]
* sample_position           vector3              [m]  ()  (0, 0, 25)
* source_position           vector3              [m]  ()  (0, 0, 0)
Data:
                          DataArrayView        <no unit>  (pixel_id)  binned data: dim='events', content=DataArray(
          dims=(events: 71300766),
          data=float64[counts],
          coords={'t':float64[s]})

<scipp.DataArray>
Dimensions: Sizes[pixel_id:98304, ]
Coordinates:
* pixel_id                    int64  [dimensionless]  (pixel_id)  [0, 1, ..., 98302, 98303]
* sample_position           vector3              [m]  ()  (0, 0, 25)
* source_position           vector3              [m]  ()  (0, 0, 0)
Data:
                          DataArrayView        <no unit>  (pixel_id)  binned data: dim='events', content=DataArray(
          dims=(events: 71300766),
          data=float64[counts],
          coords={'t':float64[s]})

In [2]:
# solid angles
d_omega = v_events.hist().data
d_omega /= d_omega.sum()
d_omega

<scipp.Variable> (pixel_id: 98304)    float64  [dimensionless]  [7.73126e-06, 7.43449e-06, ..., 7.43258e-06, 7.96688e-06]

In [ ]:
# LET banana detectors
# Horizontal -40 to 140 degs, sample to detector distance is R = 3.5 m, 384 bins
# width per pixel is w = (140+40)/180*pi*R/384 =  0.02863 m
# Vertical height is H = 4 m, 256 bins
# height per pixel is h = H/256 = 0.015625 m

import numpy as np

h = sc.norm(det_positions[384] - det_positions[0])
w = sc.norm(det_positions[1] - det_positions[0])
r = det_positions - sample_position
d = sc.norm(r)
ratio = w * h / 4 / d / sc.sqrt(d**2 + w**2 / 4 + h**2 / 4)

d_omega_calc = sc.array(dims=["pixel_id"], values=4 * np.arctan(ratio.values))

x = r.fields.x
y = r.fields.y
z = r.fields.z
rho = sc.sqrt(x**2 + z**2)
r_plus = sc.sqrt(rho**2 + w**2 / 4 + (y + h / 2) ** 2)
r_minus = sc.sqrt(rho**2 + w**2 / 4 + (y - h / 2) ** 2)
ratio_plus = w / 2 * (y + h / 2) / rho / r_plus
ratio_minus = w / 2 * (y - h / 2) / rho / r_minus
d_omega_calc_vert = sc.array(
    dims=["pixel_id"],
    values=2 * (np.arctan(ratio_plus.values) - np.arctan(ratio_minus.values)),
)


In [5]:
%matplotlib widget
import plopp as pp

pp.plot(
    {
        "Van": d_omega,
        "Calc normal (x,y,z)": d_omega_calc / d_omega_calc.sum(),
        "Calc normal (x,0,z)": d_omega_calc_vert / d_omega_calc_vert.sum(),
    },
    ylabel="fractional counts",
    title="Vanadium counts/pixel over total counts",
    linestyle="-",
    markersize=2,
    ymax=2e-5,
    ymin=0.0,
    grid="True",
)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [ ]:
pp.plot(
    {
        "Van": d_omega,
    },
    ylabel="fractional counts",
    title="Vanadium counts/pixel over total counts",
    linestyle="-",
    markersize=2,
    ymax=2e-5,
    ymin=0.0,
    grid="True",
)

In [ ]:
import numpy as np

nx, ny = 384, 256
v_square = sc.DataArray(
    data=sc.Variable(
        dims=["height", "two_theta"], values=np.reshape(d_omega.values, (ny, nx))
    ),
    coords={
        "two_theta": sc.linspace(dim="two_theta", start=-40, stop=140, num=nx),
        "height": sc.linspace(dim="height", start=-2, stop=2, num=ny),
    },
)


pp.plot(v_square, title="Vanadium square", vmin=0.5e-5, vmax=1.5e-5, cmap="turbo")

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…